In [5]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "nibabel", "scikit-image", "scipy", "matplotlib",
                       "kagglehub", "tqdm", "-q"])

import os, gc, csv, warnings
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import (binary_erosion, binary_dilation,
                            binary_opening, binary_closing,
                            label as nd_label)
from skimage.feature import canny
from skimage.filters import sobel
from skimage.morphology import disk
from tqdm import tqdm
import kagglehub

warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
MAX_VOLUMES  = 5      # number of volumes to process
MAX_SLICES   = 3      # centre slices per volume
MORPH_ITER   = 2      # morphological operation iterations
CANNY_SIGMA  = 1.5    # Canny edge detector sigma
OUT_DIR      = Path("output_a2")

for d in [OUT_DIR / "masks",
          OUT_DIR / "morphology",
          OUT_DIR / "chain_code",
          OUT_DIR / "convex_hull"]:
    d.mkdir(parents=True, exist_ok=True)

# ── DOWNLOAD DATASET ──────────────────────────────────────────────────────────
print("Downloading dataset via kagglehub...")
path = kagglehub.dataset_download("awansaad6797/cancer-dataset")
print(f"Dataset path: {path}")

all_nii = sorted(f for f in Path(path).rglob("*.nii") if f.is_file())
all_nii = [f for f in all_nii if "seg" not in f.name.lower()]
all_nii = all_nii[:MAX_VOLUMES]
print(f"Using {len(all_nii)} volumes.")

# ── HELPERS ───────────────────────────────────────────────────────────────────
def normalise(slc):
    mn, mx = slc.min(), slc.max()
    if mx > mn:
        return ((slc - mn) / (mx - mn + 1e-8)).astype(np.float32)
    return np.zeros_like(slc, dtype=np.float32)

def centre_slice_indices(n_slices, max_slices):
    if n_slices <= max_slices:
        return list(range(n_slices))
    start = (n_slices - max_slices) // 2
    return list(range(start, start + max_slices))

# ── TASK 1: SEGMENTATION (MASKING) ───────────────────────────────────────────
def canny_mask(slc, sigma=CANNY_SIGMA):
    edges  = canny(slc, sigma=sigma)
    closed = binary_closing(edges, structure=disk(4))
    labeled, n = nd_label(closed)
    if n == 0:
        return edges.astype(np.uint8), edges.astype(np.uint8)
    sizes   = [(labeled == i).sum() for i in range(1, n + 1)]
    largest = np.argmax(sizes) + 1
    mask    = (labeled == largest).astype(np.uint8)
    return edges.astype(np.uint8), mask

def sobel_mask(slc, threshold=0.1):
    magnitude = sobel(slc)
    edges     = (magnitude > threshold).astype(np.uint8)
    closed    = binary_closing(edges, structure=disk(4))
    labeled, n = nd_label(closed)
    if n == 0:
        return edges, edges
    sizes   = [(labeled == i).sum() for i in range(1, n + 1)]
    largest = np.argmax(sizes) + 1
    mask    = (labeled == largest).astype(np.uint8)
    return edges, mask

def save_mask_comparison(orig, c_edges, c_mask, s_edges, s_mask, tag):
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, (img, title) in zip(axes, [
        (orig,    "Original"),
        (c_edges, "Canny Edges"),
        (c_mask,  "Canny Mask"),
        (s_edges, "Sobel Edges"),
        (s_mask,  "Sobel Mask"),
    ]):
        ax.imshow(img, cmap="gray")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(f"Segmentation – {tag}", fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "masks" / f"{tag}_masks.png", dpi=90, bbox_inches="tight")
    plt.close(fig)

# ── TASK 2: MORPHOLOGICAL CLEANING ───────────────────────────────────────────
def morphological_pipeline(mask, iterations=MORPH_ITER):
    struct  = disk(3).astype(bool)
    eroded  = binary_erosion(mask,  structure=struct, iterations=iterations).astype(np.uint8)
    dilated = binary_dilation(mask, structure=struct, iterations=iterations).astype(np.uint8)
    opened  = binary_opening(mask,  structure=struct, iterations=iterations).astype(np.uint8)
    closed  = binary_closing(mask,  structure=struct, iterations=iterations).astype(np.uint8)
    return {
        "Original Mask": mask,
        "Erosion":       eroded,
        "Dilation":      dilated,
        "Opening":       opened,
        "Closing":       closed,
    }

def save_morphology_grid(results, tag):
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, (title, img) in zip(axes, results.items()):
        ax.imshow(img, cmap="gray")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(f"Morphological Operations – {tag}", fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "morphology" / f"{tag}_morphology.png", dpi=90, bbox_inches="tight")
    plt.close(fig)

# ── TASK 3: CHAIN CODE ────────────────────────────────────────────────────────
# Freeman 8-directional offsets
FREEMAN_DIR = {
    0: ( 0,  1),
    1: (-1,  1),
    2: (-1,  0),
    3: (-1, -1),
    4: ( 0, -1),
    5: ( 1, -1),
    6: ( 1,  0),
    7: ( 1,  1),
}

def compute_chain_code(mask):
    labeled, n = nd_label(mask)
    if n == 0:
        return None, []
    sizes   = [(labeled == i).sum() for i in range(1, n + 1)]
    region  = (labeled == (np.argmax(sizes) + 1)).astype(np.uint8)

    eroded   = binary_erosion(region, structure=np.ones((3, 3))).astype(np.uint8)
    boundary = region - eroded
    coords   = list(zip(*np.where(boundary > 0)))
    if len(coords) < 2:
        return None, []

    coord_set = set(coords)
    current   = min(coords, key=lambda p: (p[0], p[1]))
    start     = current
    chain     = []
    visited   = set()
    prev_dir  = 0

    for _ in range(len(coords) + 5):
        visited.add(current)
        found = False
        search_start = (prev_dir + 5) % 8
        for d_offset in range(8):
            d = (search_start + d_offset) % 8
            dr, dc = FREEMAN_DIR[d]
            nxt = (current[0] + dr, current[1] + dc)
            if nxt in coord_set and nxt not in visited:
                chain.append(d)
                prev_dir = d
                current  = nxt
                found    = True
                break
        if not found:
            break
        if current == start and len(chain) > 1:
            break

    return start, chain

def first_difference(chain_code):
    if len(chain_code) < 2:
        return []
    fd = [(chain_code[i] - chain_code[i - 1]) % 8 for i in range(1, len(chain_code))]
    fd.append((chain_code[0] - chain_code[-1]) % 8)
    return fd

def shape_number(first_diff):
    if not first_diff:
        return []
    doubled = first_diff * 2
    n = len(first_diff)
    min_rot = min(tuple(doubled[i:i + n]) for i in range(n))
    return list(min_rot)

def save_chain_code_viz(orig, mask, start, chain, tag):
    if not chain or start is None:
        return
    r, c = start
    path_r, path_c = [r], [c]
    for d in chain:
        dr, dc = FREEMAN_DIR[d]
        r += dr; c += dc
        path_r.append(r); path_c.append(c)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(orig, cmap="gray")
    axes[0].imshow(mask, cmap="Reds", alpha=0.4)
    axes[0].set_title("Original + Mask", fontsize=9)
    axes[0].axis("off")

    axes[1].imshow(orig, cmap="gray")
    axes[1].plot(path_c, path_r, "cyan", linewidth=1)
    axes[1].plot(start[1], start[0], "ro", markersize=5, label="Start")
    axes[1].set_title(f"Chain Code Boundary\n({len(chain)} steps)", fontsize=9)
    axes[1].legend(fontsize=7)
    axes[1].axis("off")

    fig.suptitle(f"Chain Code – {tag}", fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "chain_code" / f"{tag}_chain.png", dpi=90, bbox_inches="tight")
    plt.close(fig)

# ── TASK 4: CONVEX HULL (GRAHAM SCAN) ────────────────────────────────────────
def cross_product(O, A, B):
    return (A[0] - O[0]) * (B[1] - O[1]) - (A[1] - O[1]) * (B[0] - O[0])

def graham_scan(points):
    points = list(set(points))
    if len(points) < 3:
        return points
    pivot = min(points, key=lambda p: (p[1], p[0]))

    def polar_angle(p):
        return np.arctan2(p[1] - pivot[1], p[0] - pivot[0])

    def dist(p):
        return (p[0] - pivot[0]) ** 2 + (p[1] - pivot[1]) ** 2

    sorted_pts = sorted(
        [p for p in points if p != pivot],
        key=lambda p: (polar_angle(p), dist(p))
    )
    hull = [pivot, sorted_pts[0]]
    for p in sorted_pts[1:]:
        while len(hull) > 1 and cross_product(hull[-2], hull[-1], p) <= 0:
            hull.pop()
        hull.append(p)
    return hull

def get_boundary_points(mask, sample_every=3):
    labeled, n = nd_label(mask)
    if n == 0:
        return []
    sizes  = [(labeled == i).sum() for i in range(1, n + 1)]
    region = (labeled == (np.argmax(sizes) + 1)).astype(np.uint8)
    eroded = binary_erosion(region, structure=np.ones((3, 3))).astype(np.uint8)
    bnd    = region - eroded
    rows, cols = np.where(bnd > 0)
    return list(zip(cols[::sample_every].tolist(), rows[::sample_every].tolist()))

def save_convex_hull_viz(orig, mask, hull_pts, tag):
    if not hull_pts:
        return
    hull_closed = hull_pts + [hull_pts[0]]
    hx = [p[0] for p in hull_closed]
    hy = [p[1] for p in hull_closed]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(orig, cmap="gray")
    axes[0].imshow(mask, cmap="Reds", alpha=0.4)
    axes[0].set_title("Original + Mask", fontsize=9)
    axes[0].axis("off")

    axes[1].imshow(orig, cmap="gray")
    axes[1].imshow(mask, cmap="Blues", alpha=0.3)
    axes[1].plot(hx, hy, "y-", linewidth=2, label="Convex Hull")
    axes[1].scatter([p[0] for p in hull_pts], [p[1] for p in hull_pts],
                    c="yellow", s=15, zorder=5)
    axes[1].set_title(f"Convex Hull (Graham Scan)\n{len(hull_pts)} vertices", fontsize=9)
    axes[1].legend(fontsize=7)
    axes[1].axis("off")

    fig.suptitle(f"Convex Hull – {tag}", fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "convex_hull" / f"{tag}_hull.png", dpi=90, bbox_inches="tight")
    plt.close(fig)

# ── MAIN PIPELINE ─────────────────────────────────────────────────────────────
csv_rows = []

with open(OUT_DIR / "chain_code_report.csv", "w", newline="") as csvf:
    writer = csv.DictWriter(
        csvf,
        fieldnames=["subject", "slice", "chain_length",
                    "chain_code_preview", "first_diff_preview",
                    "shape_number_preview", "hull_vertices"]
    )
    writer.writeheader()

    for nii_path in tqdm(all_nii, desc="Volumes"):
        subj = nii_path.stem.replace(".nii", "")
        try:
            img = nib.load(str(nii_path))
            vol = img.get_fdata(dtype=np.float32)
            if vol.ndim == 4:
                vol = vol[..., 0]
            if vol.ndim != 3:
                continue

            for z in centre_slice_indices(vol.shape[2], MAX_SLICES):
                slc = normalise(vol[:, :, z])
                if slc.max() < 0.01:
                    continue

                tag = f"{subj}_z{z:04d}"

                # Task 1 – Segmentation
                c_edges, c_mask = canny_mask(slc)
                s_edges, s_mask = sobel_mask(slc)
                save_mask_comparison(slc, c_edges, c_mask, s_edges, s_mask, tag)

                primary_mask = c_mask

                # Task 2 – Morphological Cleaning
                morph_results = morphological_pipeline(primary_mask)
                save_morphology_grid(morph_results, tag)
                cleaned_mask = morph_results["Closing"]

                # Task 3 – Chain Code
                start_pt, chain = compute_chain_code(cleaned_mask)
                fd   = first_difference(chain)
                snum = shape_number(fd)
                save_chain_code_viz(slc, cleaned_mask, start_pt, chain, tag)

                # Task 4 – Convex Hull
                bnd_pts  = get_boundary_points(cleaned_mask, sample_every=3)
                hull_pts = graham_scan(bnd_pts) if len(bnd_pts) >= 3 else []
                save_convex_hull_viz(slc, cleaned_mask, hull_pts, tag)

                # CSV record
                row = {
                    "subject":              subj,
                    "slice":                z,
                    "chain_length":         len(chain),
                    "chain_code_preview":   str(chain[:20]),
                    "first_diff_preview":   str(fd[:20]),
                    "shape_number_preview": str(snum[:20]),
                    "hull_vertices":        len(hull_pts),
                }
                writer.writerow(row)
                csv_rows.append(row)

                print(f"\n  [{tag}]")
                print(f"    Chain Code (first 20)  : {chain[:20]}")
                print(f"    First Difference (f20) : {fd[:20]}")
                print(f"    Shape Number (f20)     : {snum[:20]}")
                print(f"    Hull Vertices           : {len(hull_pts)}")

            del vol
            gc.collect()

        except Exception as e:
            print(f"  [ERROR] {subj}: {e} — skipping")
            continue

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("ASSIGNMENT 2 – SUMMARY")
print(f"{'='*60}")
print(f"Slices processed          : {len(csv_rows)}")
if csv_rows:
    chain_lengths = [r["chain_length"] for r in csv_rows]
    hull_sizes    = [r["hull_vertices"] for r in csv_rows]
    print(f"Mean chain code length    : {np.mean(chain_lengths):.1f} steps")
    print(f"Mean hull vertices        : {np.mean(hull_sizes):.1f}")
print(f"\nOutputs saved to: {OUT_DIR.resolve()}")
print(f"  output_a2/masks/         — Canny & Sobel mask comparisons")
print(f"  output_a2/morphology/    — Erosion, Dilation, Opening, Closing")
print(f"  output_a2/chain_code/    — Chain code boundary overlays")
print(f"  output_a2/convex_hull/   — Graham Scan hull overlays")
print(f"  output_a2/chain_code_report.csv")
print(f"{'='*60}")

Using Colab cache for faster access to the 'cancer-dataset' dataset.
Dataset path: /kaggle/input/cancer-dataset
Using 5 volumes.


Volumes:   0%|          | 0/5 [00:00<?, ?it/s]


  [T1CE_to_SRI_defaced_z0076]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 5, 4, 4, 4]
    First Difference (f20) : [7, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0, 0, 1, 7, 0, 0, 0]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    Hull Vertices           : 53

  [T1CE_to_SRI_defaced_z0077]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 5, 4, 4, 4, 5, 4, 4, 4, 5, 4, 4, 4, 5, 4]
    First Difference (f20) : [7, 0, 0, 0, 0, 1, 7, 0, 0, 1, 7, 0, 0, 1, 7, 0, 0, 1, 7, 1]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    Hull Vertices           : 55


Volumes:  20%|██        | 1/5 [00:10<00:42, 10.55s/it]


  [T1CE_to_SRI_defaced_z0078]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 5, 4, 4, 4, 5, 4, 4, 5, 4]
    First Difference (f20) : [7, 0, 0, 0, 0, 1, 7, 0, 0, 0, 1, 7, 0, 0, 1, 7, 0, 1, 7, 0]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    Hull Vertices           : 58

  [T1_to_SRI_defaced_z0076]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 5, 4, 4, 5, 4, 4, 4, 5]
    First Difference (f20) : [7, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0, 1, 7, 0, 1, 7, 0, 0, 1, 7]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 7, 0]
    Hull Vertices           : 56

  [T1_to_SRI_defaced_z0077]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 5, 4, 4, 4, 5, 4, 4, 5, 4]
    First Difference (f20) : [7, 0, 0, 0, 0, 0, 1, 7, 0, 0, 1, 7, 0, 0, 1, 7, 0, 1, 7, 0]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 7, 0, 0]
    Hull Vertices           : 60


...


  [FL_to_SRI_defaced_z0078]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 5, 4, 4, 4, 5, 4, 4]
    First Difference (f20) : [7, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0, 1, 7, 0, 0, 1, 7, 0, 1]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    Hull Vertices           : 55

  [T2_to_SRI_defaced_z0076]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 5, 4, 4, 4, 5]
    First Difference (f20) : [7, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0, 1, 7, 0, 0, 1, 7]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    Hull Vertices           : 57

  [T2_to_SRI_defaced_z0077]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 5, 4, 4, 4, 5, 4]
    First Difference (f20) : [7, 0, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0, 1, 7, 0, 0, 1, 7, 0]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    Hull Vertices           : 57


Volumes:  80%|████████  | 4/5 [00:28<00:06,  6.58s/it]


  [T2_to_SRI_defaced_z0078]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 5, 4, 4, 4]
    First Difference (f20) : [7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0, 1, 7, 0, 0, 1]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    Hull Vertices           : 55

  [T1CE_to_SRI_defaced_z0076]
    Chain Code (first 20)  : [5, 4, 4, 4, 5, 4, 4, 5, 4, 4, 5, 4, 5, 4, 4, 5, 4, 5, 4, 4]
    First Difference (f20) : [7, 0, 0, 1, 7, 0, 1, 7, 0, 1, 7, 1, 7, 0, 1, 7, 1, 7, 0, 1]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0]
    Hull Vertices           : 46

  [T1CE_to_SRI_defaced_z0077]
    Chain Code (first 20)  : [5, 4, 4, 4, 5, 4, 4, 5, 4, 4, 5, 4, 5, 4, 4, 5, 4, 5, 4, 4]
    First Difference (f20) : [7, 0, 0, 1, 7, 0, 1, 7, 0, 1, 7, 1, 7, 0, 1, 7, 1, 7, 0, 1]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 0, 0]
    Hull Vertices           : 45


Volumes: 100%|██████████| 5/5 [00:34<00:00,  6.84s/it]


  [T1CE_to_SRI_defaced_z0078]
    Chain Code (first 20)  : [5, 4, 4, 4, 4, 5, 4, 5, 4, 4, 5, 4, 5, 4, 4, 5, 4, 5, 4, 4]
    First Difference (f20) : [7, 0, 0, 0, 1, 7, 1, 7, 0, 1, 7, 1, 7, 0, 1, 7, 1, 7, 0, 1]
    Shape Number (f20)     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7, 0, 0, 7, 1, 1, 7]
    Hull Vertices           : 46

ASSIGNMENT 2 – SUMMARY
Slices processed          : 15
Mean chain code length    : 469.5 steps
Mean hull vertices        : 54.0

Outputs saved to: /content/output_a2
  output_a2/masks/         — Canny & Sobel mask comparisons
  output_a2/morphology/    — Erosion, Dilation, Opening, Closing
  output_a2/chain_code/    — Chain code boundary overlays
  output_a2/convex_hull/   — Graham Scan hull overlays
  output_a2/chain_code_report.csv


In [6]:
!zip -r output_a2.zip /content/output_a2

updating: content/output_a2/ (stored 0%)
updating: content/output_a2/masks/ (stored 0%)
updating: content/output_a2/masks/T1_to_SRI_defaced_z0078_masks.png (deflated 9%)
updating: content/output_a2/masks/T1_to_SRI_defaced_z0077_masks.png (deflated 9%)
updating: content/output_a2/masks/FL_to_SRI_defaced_z0076_masks.png (deflated 8%)
updating: content/output_a2/masks/T2_to_SRI_defaced_z0077_masks.png (deflated 9%)
updating: content/output_a2/masks/T1CE_to_SRI_defaced_z0077_masks.png (deflated 10%)
updating: content/output_a2/masks/T2_to_SRI_defaced_z0076_masks.png (deflated 9%)
updating: content/output_a2/masks/T1CE_to_SRI_defaced_z0076_masks.png (deflated 9%)
updating: content/output_a2/masks/T1_to_SRI_defaced_z0076_masks.png (deflated 9%)
updating: content/output_a2/masks/FL_to_SRI_defaced_z0078_masks.png (deflated 8%)
updating: content/output_a2/masks/T2_to_SRI_defaced_z0078_masks.png (deflated 9%)
updating: content/output_a2/masks/T1CE_to_SRI_defaced_z0078_masks.png (deflated 10%)
up